In [ ]:
2+2

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np

In [ ]:

class Tokenizer():
    def __init__(self):
        pass

    def tokenize(self, text):
        text = text.lower()
        text = text.replace('?', '')
        text = text.replace("'", '')
        return text.split()    



In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim=512, max_seq_len=5000):
        super().__init__()

        # Create matrix
        pe = torch.zeros(max_seq_len, embedding_dim)

        # Position: 0, 1, 2, 3, ...
        position = torch.arange(0,max_seq_len,dtype=torch.float).unsqueeze(1)

        # Division term
        div_term = torch.exp(torch.arange(0,embedding_dim,2).float()* (-math.log(10000.0) / embedding_dim))

        # Even dimensions -> sin
        pe[:, 0::2] = torch.sin(position * div_term)

        # Odd dimensions -> cos
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Register as buffer
        self.register_buffer('pe', pe)


    def forward(self, x):

        # x shape:
        # (sequence_length, embedding_dim)

        seq_len = x.size(0)

        positional_encoding = self.pe[0,:seq_len,:]

        return positional_encoding

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads 
        self.embedding_dim = embedding_dim 

        # dimension of each head
        self.head_dim = embedding_dim // num_heads 

        # Q, K, V
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, input_features):  


        Q = self.W_Q(input_features)  
        K = self.W_K(input_features)  
        V = self.W_V(input_features) 

        seq_len = input_features.shape[0] 

        Q = Q.reshape(seq_len,self.num_heads,self.head_dim) 
        K = K.reshape(seq_len,self.num_heads,self.head_dim) 
        V = V.reshape(seq_len,self.num_heads,self.head_dim) 

        heads = []
        for head in range(self.num_heads):
            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1) 
        output = self.W_O(multi_head) 
        return output








In [ ]:

class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads
        self.embedding_dim = embedding_dim

        # dimension of each head
        self.head_dim = embedding_dim // num_heads

        # Q, K, V
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, input_features):

        Q = self.W_Q(input_features)
        K = self.W_K(input_features)
        V = self.W_V(input_features)

        seq_len = input_features.shape[0]

        Q = Q.reshape(seq_len, self.num_heads, self.head_dim)
        K = K.reshape(seq_len, self.num_heads, self.head_dim)
        V = V.reshape(seq_len, self.num_heads, self.head_dim)

        heads = []

        for head in range(self.num_heads):

            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # =====================
            # Causal Mask
            # =====================

            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=input_features.device),
                diagonal=1
            )

            scores = scores.masked_fill(mask == 1, float('-inf'))

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1)

        output = self.W_O(multi_head)

        return output



In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim=512, ff_dim=2048):
        super().__init__()

        # First projection: 512 -> 2048
        self.linear1 = nn.Linear(embedding_dim, ff_dim)

        # ReLU activation
        self.relu = nn.ReLU()

        # Second projection: 2048 -> 512
        self.linear2 = nn.Linear(ff_dim, embedding_dim)

    def forward(self, x):
        # x shape: (sequence_length, 512)

        x = self.linear1(x)   # (seq_len, 2048)
        x = self.relu(x)      # (seq_len, 2048)
        x = self.linear2(x)   # (seq_len, 512)

        return x



In [ ]:
class WordEmbedding(nn.Module):

    def __init__(self, vocab_size, embedding_dim=512):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embedding_dim)

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)
        return embeddings



In [ ]:
class CrossMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads
        self.embedding_dim = embedding_dim

        # dimension of each head
        self.head_dim = embedding_dim // num_heads

        # Query comes from decoder
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)

        # Key and Value come from encoder
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, decoder_features, encoder_output):

        # Query from decoder
        Q = self.W_Q(decoder_features)

        # Key and Value from encoder
        K = self.W_K(encoder_output)
        V = self.W_V(encoder_output)

        decoder_seq_len = decoder_features.shape[0]
        encoder_seq_len = encoder_output.shape[0]

        Q = Q.reshape(
            decoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.reshape(
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.reshape(
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        heads = []

        for head in range(self.num_heads):

            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1)

        output = self.W_O(multi_head)

        return output



In [ ]:
class AddAndNormalize(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()

        self.layer_norm = nn.LayerNorm(embedding_dim)

    def forward(self, x, sublayer_output):
        # Add residual connection
        x = x + sublayer_output

        # Normalize
        x = self.layer_norm(x)

        return x



In [ ]:

class Encoder(nn.Module):

    def __init__(
        self,
        num_layers=6,
        embedding_dim=512,
        num_heads=16,
        ff_dim=2048
    ):
        super().__init__()

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim
        )

        self.multi_head_attention = MultiHeadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads
        )

        self.add_norm = AddAndNormalize(
            embedding_dim=embedding_dim
        )

        self.feed_forward = FeedForward(
            embedding_dim=embedding_dim,
            ff_dim=ff_dim
        )

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

    def forward(self, embeddings):

        positional_encoding = self.positional_encoding(
            embeddings
        )

        x = embeddings + positional_encoding

        for _ in range(self.num_layers):

            attention_output = self.multi_head_attention(
                x
            )

            attention_output = self.add_norm(
                x,
                attention_output
            )

            feed_forward_output = self.feed_forward(
                attention_output
            )

            x = self.add_norm(
                attention_output,
                feed_forward_output
            )

        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16, ff_dim=2048):
        super().__init__()

        self.masked_multi_head_attention = MaskedMultiHeadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads
        )

        self.feed_forward = FeedForward(
            embedding_dim=embedding_dim,
            ff_dim=ff_dim
        )

        # Separate norm layers for each residual connection
        self.add_norm_attention = AddAndNormalize(embedding_dim=embedding_dim)
        self.add_norm_ff = AddAndNormalize(embedding_dim=embedding_dim)

    def forward(self, x):

        attention_output = self.masked_multi_head_attention(x)
        x = self.add_norm_attention(x, attention_output)

        feed_forward_output = self.feed_forward(x)
        x = self.add_norm_ff(x, feed_forward_output)

        return x


class Decoder(nn.Module):

    def __init__(
        self,
        num_layers=6,
        embedding_dim=512,
        num_heads=16,
        ff_dim=2048
    ):
        super().__init__()

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim
        )

        # Independent layers, each with its own weights
        self.layers = nn.ModuleList([
            DecoderLayer(embedding_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ])

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

    def forward(self, embeddings):

        positional_encoding = self.positional_encoding(embeddings)

        x = embeddings + positional_encoding

        for layer in self.layers:
            x = layer(x)

        return x

In [ ]:
import os

# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# TRAINING DATA
# =========================================================

input_path = "/kaggle/input/datasets/prabhavsinghal/text-data/input.txt"

with open(input_path, "r", encoding="utf-8") as file:
    text = file.read()

texts = [line.strip() for line in text.splitlines() if line.strip()]

print("Number of training lines:", len(texts))


# =========================================================
# TOKENIZER
# =========================================================

tokenizer = Tokenizer()


# =========================================================
# BUILD VOCABULARY
# =========================================================

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}

for line in texts:
    for token in tokenizer.tokenize(line):
        if token not in vocab:
            vocab[token] = len(vocab)

id_to_token = {index: token for token, index in vocab.items()}
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)


# =========================================================
# MODEL SETTINGS
# =========================================================

embedding_dim = 64
num_heads = 4
ff_dim = 128
num_layers = 2


# =========================================================
# DECODER-ONLY MODEL
# =========================================================

decoder = Decoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
).to(device)

decoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
).to(device)

output_layer = nn.Linear(embedding_dim, vocab_size).to(device)


# =========================================================
# LOSS / OPTIMIZER
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(decoder.parameters())
    + list(decoder_embedding.parameters())
    + list(output_layer.parameters()),
    lr=0.001
)


# =========================================================
# TRAINING
# =========================================================

epochs = 1000

for epoch in range(epochs):

    total_loss = 0.0
    num_sequences = 0

    for line in texts:

        tokens = tokenizer.tokenize(line)

        if len(tokens) == 0:
            continue

        # Full sequence: <BOS> tok1 tok2 ... tokN <EOS>
        token_ids = (
            [vocab["<BOS>"]]
            + [vocab.get(t, vocab["<UNK>"]) for t in tokens]
            + [vocab["<EOS>"]]
        )

        if len(token_ids) < 2:
            continue

        sequence = torch.tensor(token_ids, dtype=torch.long, device=device)

        # Standard causal LM setup: predict next token at every position
        input_ids = sequence[:-1]
        target_ids = sequence[1:]

        embeddings = decoder_embedding(input_ids)
        decoder_output = decoder(embeddings)
        logits = output_layer(decoder_output)

        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_sequences += 1

    average_loss = total_loss / max(num_sequences, 1)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs} Loss: {average_loss:.4f}")


# =========================================================
# SAVE MODEL
# =========================================================

checkpoint = {
    "vocab": vocab,
    "decoder": decoder.state_dict(),
    "decoder_embedding": decoder_embedding.state_dict(),
    "output_layer": output_layer.state_dict(),
    "embedding_dim": embedding_dim,
    "num_heads": num_heads,
    "ff_dim": ff_dim,
    "num_layers": num_layers
}

model_path = "/kaggle/working/model.pth"
torch.save(checkpoint, model_path)

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print("Model saved to:", model_path)
print("Model size:", f"{os.path.getsize(model_path) / (1024 ** 2):.2f} MB")

In [ ]:
import torch
import torch.nn as nn

# =========================================================
# CONFIGURATION
# =========================================================

CHECKPOINT_PATH = "/kaggle/working/model.pth"

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = Tokenizer()

# =========================================================
# LOAD CHECKPOINT
# =========================================================

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

vocab = checkpoint["vocab"]
id_to_token = {index: token for token, index in vocab.items()}
vocab_size = len(vocab)

embedding_dim = checkpoint["embedding_dim"]
num_heads = checkpoint["num_heads"]
ff_dim = checkpoint["ff_dim"]
num_layers = checkpoint["num_layers"]

# =========================================================
# MODEL
# =========================================================

decoder = Decoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)

decoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)

output_layer = nn.Linear(embedding_dim, vocab_size)

decoder.load_state_dict(checkpoint["decoder"])
decoder_embedding.load_state_dict(checkpoint["decoder_embedding"])
output_layer.load_state_dict(checkpoint["output_layer"])

decoder.eval()
decoder_embedding.eval()
output_layer.eval()

# =========================================================
# INFERENCE
# =========================================================

def generate(text, max_length=10):

    tokens = tokenizer.tokenize(text)

    if len(tokens) == 0:
        return ""

    # Seed the sequence with <BOS> + the prompt tokens
    input_ids = [vocab["<BOS>"]] + [
        vocab.get(t, vocab["<UNK>"]) for t in tokens
    ]

    generated_tokens = []

    with torch.no_grad():

        for _ in range(max_length):

            sequence = torch.tensor(input_ids, dtype=torch.long)

            embeddings = decoder_embedding(sequence)
            decoder_output = decoder(embeddings)
            logits = output_layer(decoder_output)

            last_logits = logits[-1]
            next_token_id = torch.argmax(last_logits).item()

            if next_token_id == vocab["<EOS>"]:
                break

            if next_token_id not in [vocab["<PAD>"], vocab["<BOS>"]]:
                generated_tokens.append(id_to_token[next_token_id])

            input_ids.append(next_token_id)

    return " ".join(generated_tokens)


# =========================================================
# INTERACTIVE INFERENCE
# =========================================================

if __name__ == "__main__":

    print("Model loaded.")
    print("Type 'quit' to exit.")
    print()

    while True:
        text = input("Input: ").strip()
        if text.lower() == "quit":
            break
        result = generate(text, max_length=10)
        print("Generated:", result)